## 环境准备

In [ ]:
import sys

sys.path.append("..")

In [ ]:
import json
from collections import Counter
from pathlib import Path
from typing import Literal

import jieba
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset

## 超参数

### 路径

In [ ]:
ROOT = Path.cwd().parent


def get_dir(dir: Path) -> Path:
    if not dir.exists():
        dir.mkdir(parents=True)
    return dir


DATASETS_DIR = get_dir(ROOT / "datasets")
OUTPUTS_DIR = get_dir(ROOT / "outputs")

RAW_DATASETS_DIR = get_dir(DATASETS_DIR / "raw")
PROCESSED_DATASETS_DIR = get_dir(DATASETS_DIR / "processed")

MODELS_DIR = get_dir(OUTPUTS_DIR / "models")
LOGS_DIR = get_dir(OUTPUTS_DIR / "logs")
CHECKPOINTS_DIR = get_dir(OUTPUTS_DIR / "checkpoints")
FIGURES_DIR = get_dir(OUTPUTS_DIR / "figures")

STOPWORDS_PATH = DATASETS_DIR / "stopwords.txt"
RAW_DATASETS_PATH = RAW_DATASETS_DIR / "online_shopping_10_cats.csv"
PROCESSED_DATASETS_PATH = PROCESSED_DATASETS_DIR / "processed_datasets.csv"

BEST_MODEL_PATH = MODELS_DIR / "best_model.pth"
LAST_MODEL_PATH = MODELS_DIR / "last_model.pth"

### 数据

In [ ]:
# 最长序列长度
MAX_SEQ_LENGTH = 128

# 最小频率
MIN_FREQ = 3

# 是否移除停用词
REMOVE_STOPWORDS = True

# 训练集, 验证集, 测试集的比例
TRAIN_RATIO = 0.7
VALID_RATIO = 0.2
TEST_RATIO = 0.1

### 模型

In [ ]:
# 模型类型
MODEL_TYPE: Literal["LSTM", "RNN", "GRU"] = "LSTM"

# 嵌入维度
EMBEDDING_DIM = 300

# 隐藏层维度
HIDDEN_DIM = 256

# 层数数量
NUM_LAYERS = 2

# Dropout率
DROPOUT = 0.5

# 是否使用双向RNN
BIDIRECTIONAL = True

# 使用Attention机制
USE_ATTENTION = True

### 训练

In [ ]:
# Batch大小
BATCH_SIZE = 64

# 学习率
LEARNING_RATE = 1e-3

# 权重衰减
WEIGHT_DECAY = 1e-5

# 训练轮数
EPOCHS = 50

# 梯度裁剪
GRAD_CLIP = 1.0

# 学习率调整参数
# 当验证集性能不再提升时，学习率乘以LR_REDUCE_FACTOR
LR_REDUCE_FACTOR = 0.5
# 当验证集性能不再提升时，等待LR_REDUCE_PATIENCE轮后才调整学习率
LR_REDUCE_PATIENCE = 3

# 早停参数
# 当验证集性能不再提升时，等待EARLY_STOP_PATIENCE轮后停止训练
EARLY_STOP_PATIENCE = 5
# 当验证集性能提升小于EARLY_STOP_MIN_DELTA时，认为性能不再提升
EARLY_STOP_MIN_DELTA = 1e-4

## 数据处理

### 词表映射

In [ ]:
class VocabMapping:
    def __init__(self, word2idx: dict[str, int], idx2word: dict[int, str]):
        self.word2idx = word2idx
        self.idx2word = idx2word

    @property
    def vocab_size(self) -> int:
        return len(self.word2idx)

    @property
    def pad_idx(self) -> int:
        return 0

    @property
    def unk_idx(self) -> int:
        return 1

    def to_dict(self) -> dict[str, dict]:
        return {
            "word2idx": self.word2idx,
            "idx2word": {str(k): v for k, v in self.idx2word.items()},
        }

    @classmethod
    def from_dict(cls, data: dict[str, dict]) -> "VocabMapping":
        word2idx = data["word2idx"]
        idx2word = {int(k): v for k, v in data["idx2word"].items()}
        return cls(word2idx, idx2word)

    def save(self, path: Path) -> None:
        with path.open("w", encoding="utf-8") as f:
            json.dump(self.to_dict(), f, ensure_ascii=False, indent=4)

    @classmethod
    def load(cls, path: Path) -> "VocabMapping":
        with path.open("r", encoding="utf-8") as f:
            data = json.load(f)
            return cls.from_dict(data)

### 数据预处理

In [ ]:
df = pd.read_csv(RAW_DATASETS_PATH)
df

,Unnamed: 0,cat,label,text
0,0,书籍,1,﻿做父母一定要有刘墉这样的心态，不断地学习，不断地进步，不断地给自己补充新鲜血液，让自己保持...
1,1,书籍,1,作者真有英国人严谨的风格，提出观点、进行论述论证，尽管本人对物理学了解不深，但是仍然能感受到...
2,2,书籍,1,作者长篇大论借用详细报告数据处理工作和计算结果支持其新观点。为什么荷兰曾经县有欧洲最高的生产...
3,3,书籍,1,作者在战几时之前用了＂拥抱＂令人叫绝．日本如果没有战败，就有会有美军的占领，没胡官僚主义的延...
4,4,书籍,1,作者在少年时即喜阅读，能看出他精读了无数经典，因而他有一个庞大的内心世界。他的作品最难能可贵...
...,...,...,...,...
62769,62769,酒店,0,我们去盐城的时候那里的最低气温只有4度，晚上冷得要死，居然还不开空调，投诉到酒店客房部，得到...
62770,62770,酒店,0,房间很小，整体设施老化，和四星的差距很大。毛巾太破旧了。早餐很简陋。房间隔音很差，隔两间房间...
62771,62771,酒店,0,我感觉不行。。。性价比很差。不知道是银川都这样还是怎么的！
62772,62772,酒店,0,房间时间长，进去有点异味！服务员是不是不够用啊！我在一楼找了半个小时以上才找到自己房间，想找...


In [ ]:
def read_raw(filepath: Path = RAW_DATASETS_PATH) -> tuple[list[str], list[int]]:
    """
    读取原始数据集文件，返回一个包含文本和标签的列表。
    Args:
        filepath (Path): 原始数据集文件的路径
    Returns:
        texts (list[str]): 文本列表
        labels (list[int]): 标签列表
    """
    if filepath.suffix == ".csv":
        df = pd.read_csv(filepath)
    elif filepath.suffix == ".json":
        df = pd.read_json(filepath)
    else:
        raise ValueError(f"不支持的文件格式: {str(filepath)}")

    texts = [
        str(t).strip() for t in df["text"].tolist() if pd.notna(t) and str(t).strip()
    ]
    labels = [
        l
        for i, l in enumerate(df["label"].tolist())  # noqa: E741
        if pd.notna(df["text"].iloc[i]) and str(df["text"].iloc[i]).strip()
    ]

    # 过滤空文本和非法标签
    valid = [(t, l) for t, l in zip(texts, labels) if t and (l in (0, 1))]  # noqa: E741
    texts = [v[0] for v in valid]
    labels = [v[1] for v in valid]

    print(f"读取数据: {len(texts)}条")
    print(f"正样本: {sum(labels)}, 负样本: {len(labels) - sum(labels)}")

    return texts, labels

### 分词

In [ ]:
def tokenize(
    texts: list[str], remove_stopwords: bool = REMOVE_STOPWORDS
) -> list[list[str]]:
    """
    jieba 分词, 去掉停用词
    Args:
        texts(list[str]): 文本列表
        remove_stopwords(bool): 是否移除停用词
    Returns:
        result: 分词结果

    """
    stopwords = set()
    stopwords_path = STOPWORDS_PATH
    if remove_stopwords and stopwords_path.exists():
        with stopwords_path.open(mode="r", encoding="utf-8") as f:
            stopwords = set(line.strip for line in f if line.strip())

    result = []
    for text in texts:
        words = jieba.lcut(text)
        words = [w.strip() for w in words if w.strip()]
        if remove_stopwords:
            words = [w for w in words if w not in stopwords]
        if words:
            result.append(words)

    print(f"分词成功: {len(result)}")
    print(f"平均长度为: {np.mean([len(r) for r in result])}")
    return result

### 构建词表

In [ ]:
def build_vocab(
    tokenized_texts: list[list[str]], min_freq: int = MIN_FREQ
) -> VocabMapping:
    """
    构建词表
    Args:
        tokenized_texts(list[str]): 分词后的文本列表
        min_freq(int): 词出现的最小频率
    Returns:
        VocabMapping(word2idx, idx2word): 分词表
    """
    counter = Counter()
    for tokens in tokenized_texts:
        counter.update(tokens)

    word2idx = {"<PAD>": 0, "<UNK>": 1}
    idx2word = {0: "<PAD>", 1: "<UNK>"}

    idx = 2
    for word, freq in counter.most_common():
        if freq >= min_freq:
            word2idx[word] = idx
            idx2word[idx] = word
            idx += 1

    print(f"词表大小: {len(word2idx)} (min_freq = {min_freq})")
    return VocabMapping(word2idx, idx2word)

### 文本 -> 索引序列

In [ ]:
def encode(
    tokenized_texts: list[list[str]], word2idx: dict[str, int]
) -> list[list[str]]:
    """
    文本变成索引序列
    Args:
        tokenized_texts(list[list[str]]): 被分词后的文本
        word2idx(dict[str, int]): 词表索引
    Returns:
        sequences(list[list[str]]): 索引列表
    """
    sequences = []
    for tokens in tokenized_texts:
        seq = [word2idx.get(w, 1) for w in tokens]
        sequences.append(seq)

    return sequences

### 填充和截断

In [ ]:
def pad_or_truncate(
    sequences: list[list[str]], max_seq_len: int = MAX_SEQ_LENGTH
) -> torch.Tensor:
    """
    填充和截断一整个sequences
    Args:
        sequences(list[list[str]]): 索引列表
        max_seq_len(int): 截断长度
    """
    padded = []
    for seq in sequences:
        if len(seq) > max_seq_len:
            padded.append(seq[:max_seq_len])
        else:
            padded.append(seq + [0] * (max_seq_len - len(seq)))
    return torch.tensor(padded, dtype=torch.long)

### 生成Attension Mask

In [ ]:
def create_mask(padded: torch.Tensor) -> torch.Tensor:
    """
    生成Attention Mask
    Args:
        padded(torch.Tensor): 被处理过的文本序列
    Returns:
        Tensor: Attention 掩码
    """
    return (padded != 0).long()

### 划分数据集

In [ ]:
def split_data(
    input_idxs: list[int], masks: list[torch.Tensor], labels: list[int]
) -> tuple[
    tuple[torch.Tensor, torch.Tensor, torch.Tensor],
    tuple[torch.Tensor, torch.Tensor, torch.Tensor],
    tuple[torch.Tensor, torch.Tensor, torch.Tensor],
]:
    """
    划分 train/val/test
    Args:
        input_idxs: 输入的文本索引
        masks: Attention Mask
        labels: 标签
    Returns:
        ((train_idxs, train_masks, train_labels),
        (val_idxs, val_masks, val_labels),
        (test_idxs, test_masks, test_labels))
    """
    labels_tensor = torch.tensor(labels, dtype=torch.float32)

    temp_ratio = VALID_RATIO + TEST_RATIO

    # stratify 参数确保了训练集、验证集和测试集中正负样本的比例与原始数据集相同
    train_idxs, temp_idxs, train_masks, temp_masks, train_labels, temp_labels = (
        train_test_split(
            input_idxs,
            masks,
            labels_tensor,
            test_size=temp_ratio,
            stratify=labels_tensor.numpy(),
            random_state=42,
        )
    )

    # 计算验证集在临时数据中的比例
    val_ratio_in_temp = VALID_RATIO / temp_ratio
    val_idxs, test_idxs, val_masks, test_masks, val_labels, test_labels = (
        train_test_split(
            temp_idxs,
            temp_masks,
            temp_labels,
            test_size=1 - val_ratio_in_temp,
            stratify=temp_labels.numpy(),
            random_state=42,
        )
    )

    print(f"划分: train={len(train_idxs)}, val={len(val_idxs)}, test={len(test_idxs)}")
    return (
        (train_idxs, train_masks, train_labels),
        (val_idxs, val_masks, val_labels),
        (test_idxs, test_masks, test_labels),
    )

### 定义数据集类

In [ ]:
class SentimentDataset(Dataset):
    def __init__(self, input_idxs, masks, labels):
        self.input_idxs = input_idxs
        self.masks = masks
        self.labels = labels.float().unsqueeze(1)

    def __getitem__(self, idx):
        return self.input_idxs[idx], self.masks[idx], self.labels[idx]

    def __len__(self):
        return len(self.labels)

### 获取DataLoader

In [ ]:
def create_data_loaders(
    train_data, val_data, test_data, batch_size=BATCH_SIZE, num_workers=4
):
    """创建 DataLoader"""
    train_ds = SentimentDataset(*train_data)
    val_ds = SentimentDataset(*val_data)
    test_ds = SentimentDataset(*test_data)

    train_loader = DataLoader(
        train_ds, batch_size=batch_size, shuffle=True, num_workers=num_workers
    )
    val_loader = DataLoader(
        val_ds, batch_size=batch_size, shuffle=False, num_workers=num_workers
    )
    test_loader = DataLoader(
        test_ds, batch_size=batch_size, shuffle=False, num_workers=num_workers
    )

    return train_loader, val_loader, test_loader

### 使用真实数据

## 模型定义

### Bahdanau 加性注意力

### BaseRNN 基类

### 创建模型

## 训练组件

### 早停

### 保存和加载checkpoint

## 训练

### 训练一个epoch

### 评测一个epoch

### 完整流程

## 可视化

## 测试集评估

## 混淆矩阵

## ROC 曲线

## 推理